In [1]:
import os
import json
import pandas as pd
import traceback

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
KEY=os.getenv("GOOGLE_API_KEY")

In [5]:
llm = ChatGoogleGenerativeAI(google_api_key=KEY, model="gemini-2.5-flash", temperature=0.5, max_output_tokens=2048)

In [6]:
llm

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', temperature=0.5, max_output_tokens=2048, client=<google.genai.client.Client object at 0x0000024BCB581BE0>, default_metadata=(), model_kwargs={})

In [7]:
from langchain_core.prompts import PromptTemplate

In [8]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [9]:
TEMPLATE = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [10]:
quiz_generation_prompt=PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [11]:
quiz_chain = quiz_generation_prompt | llm

In [12]:
TEMPLATE2="""
You are an expert {subject} teacher.

Here is a quiz:
{quiz}

Your task:
- Review each question
- Fix any incorrect questions or answers
- Improve clarity
- Ensure correctness

Return ONLY the corrected MCQs in the same format.
"""

In [13]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [14]:
from langchain_core.runnables import RunnablePassthrough

In [15]:
generate_evaluate_chain=(
    {
        "quiz": quiz_chain,                     # output of first chain
        "subject": RunnablePassthrough()   # pass subject forward
    }
    | quiz_evaluation_prompt
    | llm
)

In [34]:
file_path=r"..\data.txt"

In [35]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [36]:
TEXT

'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]\n\nThe earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] Hebb\'s model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.[9] Other researchers who have studied human cognitive systems co

In [19]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [20]:
NUMBER=5 
SUBJECT="machine learning"
TONE="simple"

In [21]:
from langchain_core.globals import set_verbose
set_verbose(True)

In [22]:
response = generate_evaluate_chain.invoke({
    "text": TEXT,
    "number": NUMBER,
    "subject":SUBJECT,
    "tone": TONE,
    "response_json": json.dumps(RESPONSE_JSON)
})

In [23]:
response

AIMessage(content='```json\n{\n"1": {\n"mcq": "Who is credited with coining the term \'machine learning\' in 1959?",\n"options": {\n"a": "Donald Hebb",\n"b": "Arthur Samuel",\n"c": "Tom M. Mitchell",\n"d": "Ian Goodfellow"\n},\n"correct": "b"\n},\n"2": {\n"mcq": "What was the primary function of the experimental \'learning machine\' called Cybertron, developed by Raytheon Company in the early 1960s?",\n"options": {\n"a": "To play checkers against human opponents",\n"b": "To publish books on pattern classification",\n"c": "To analyze sonar signals, electrocardiograms, and speech patterns",\n"d": "To develop generative adversarial networks"\n},\n"correct": "c"\n},\n"3": {\n"mcq": "According to Tom M. Mitchell\'s widely quoted definition, what indicates that a computer program has \'learned\'?",\n"options": {\n"a": "It can perfectly imitate human responses to questions.",\n"b": "Its performance at tasks improves with experience.",\n"c": "It can identify 40 different characters from a term

In [24]:
usage = response.usage_metadata

print("Prompt Tokens:", usage.get("input_tokens"))
print("Completion Tokens:", usage.get("output_tokens"))
print("Total Tokens:", usage.get("total_tokens"))

Prompt Tokens: 1680
Completion Tokens: 1384
Total Tokens: 3064


In [25]:
print(response.content)

```json
{
"1": {
"mcq": "Who is credited with coining the term 'machine learning' in 1959?",
"options": {
"a": "Donald Hebb",
"b": "Arthur Samuel",
"c": "Tom M. Mitchell",
"d": "Ian Goodfellow"
},
"correct": "b"
},
"2": {
"mcq": "What was the primary function of the experimental 'learning machine' called Cybertron, developed by Raytheon Company in the early 1960s?",
"options": {
"a": "To play checkers against human opponents",
"b": "To publish books on pattern classification",
"c": "To analyze sonar signals, electrocardiograms, and speech patterns",
"d": "To develop generative adversarial networks"
},
"correct": "c"
},
"3": {
"mcq": "According to Tom M. Mitchell's widely quoted definition, what indicates that a computer program has 'learned'?",
"options": {
"a": "It can perfectly imitate human responses to questions.",
"b": "Its performance at tasks improves with experience.",
"c": "It can identify 40 different characters from a terminal.",
"d": "It successfully uses model-based method

In [26]:
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser()

In [27]:
quiz=parser.parse(response.content)

In [28]:
quiz

{'1': {'mcq': "Who is credited with coining the term 'machine learning' in 1959?",
  'options': {'a': 'Donald Hebb',
   'b': 'Arthur Samuel',
   'c': 'Tom M. Mitchell',
   'd': 'Ian Goodfellow'},
  'correct': 'b'},
 '2': {'mcq': "What was the primary function of the experimental 'learning machine' called Cybertron, developed by Raytheon Company in the early 1960s?",
  'options': {'a': 'To play checkers against human opponents',
   'b': 'To publish books on pattern classification',
   'c': 'To analyze sonar signals, electrocardiograms, and speech patterns',
   'd': 'To develop generative adversarial networks'},
  'correct': 'c'},
 '3': {'mcq': "According to Tom M. Mitchell's widely quoted definition, what indicates that a computer program has 'learned'?",
  'options': {'a': 'It can perfectly imitate human responses to questions.',
   'b': 'Its performance at tasks improves with experience.',
   'c': 'It can identify 40 different characters from a terminal.',
   'd': 'It successfully use

In [30]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [31]:
quiz_table_data

[{'MCQ': "Who is credited with coining the term 'machine learning' in 1959?",
  'Choices': 'a: Donald Hebb | b: Arthur Samuel | c: Tom M. Mitchell | d: Ian Goodfellow',
  'Correct': 'b'},
 {'MCQ': "What was the primary function of the experimental 'learning machine' called Cybertron, developed by Raytheon Company in the early 1960s?",
  'Choices': 'a: To play checkers against human opponents | b: To publish books on pattern classification | c: To analyze sonar signals, electrocardiograms, and speech patterns | d: To develop generative adversarial networks',
  'Correct': 'c'},
 {'MCQ': "According to Tom M. Mitchell's widely quoted definition, what indicates that a computer program has 'learned'?",
  'Choices': 'a: It can perfectly imitate human responses to questions. | b: Its performance at tasks improves with experience. | c: It can identify 40 different characters from a terminal. | d: It successfully uses model-based methods for decision making.',
  'Correct': 'b'},
 {'MCQ': 'Which 

In [32]:
df=pd.DataFrame(quiz_table_data)

In [33]:
df

,MCQ,Choices,Correct
0,Who is credited with coining the term 'machine...,a: Donald Hebb | b: Arthur Samuel | c: Tom M. ...,b
1,What was the primary function of the experimen...,a: To play checkers against human opponents | ...,c
2,According to Tom M. Mitchell's widely quoted d...,a: It can perfectly imitate human responses to...,b
3,Which of the following is NOT listed as an obj...,a: Clustering | b: Regression | c: Dimensional...,b
4,What significant contribution did Canadian psy...,a: He invented the first computer program to c...,c


In [38]:
df.to_csv("Machine_Learning_quiz.csv",index=False)